# Occupational Accident Prediction
## Application of Optimized Machine Learning Techniques

**Pipeline stages covered in this notebook:**
1. Data Loading & Overview
2. Exploratory Data Analysis (EDA)
3. Preprocessing & Feature Engineering
4. Baseline Model Training
5. Hyperparameter Optimisation
6. Model Evaluation & Comparison
7. Feature Importance & SHAP
8. Single-Record Prediction Demo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import sys
sys.path.insert(0, str(Path('..').resolve()))

plt.rcParams['figure.dpi'] = 120
print('Environment ready.')

## 1 · Data Loading & Overview

In [ ]:
from src.generate_dataset import generate_dataset

df = generate_dataset(n=14820)
print(f'Shape: {df.shape}')
print(f'Accident rate: {df["accident_occurred"].mean():.1%}')
df.head()

In [ ]:
# Data types & missing values
info = pd.DataFrame({
    'dtype':   df.dtypes,
    'nulls':   df.isnull().sum(),
    'unique':  df.nunique(),
    'sample':  df.iloc[0],
})
info

In [ ]:
df.describe().round(2)

## 2 · Exploratory Data Analysis

In [ ]:
# Accident rate by industry
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Accident Rate by Key Features', fontsize=14, fontweight='bold')

cat_cols = ['industry_sector','shift_type','ppe_compliance',
            'experience_level','age_group','employment_type']

for ax, col in zip(axes.flat, cat_cols):
    rates = df.groupby(col)['accident_occurred'].mean().sort_values(ascending=False)
    colors = ['#E24B4A' if v > df['accident_occurred'].mean() else '#185FA5'
              for v in rates.values]
    rates.plot(kind='bar', ax=ax, color=colors, edgecolor='none')
    ax.axhline(df['accident_occurred'].mean(), color='k',
               linestyle='--', linewidth=1, label='Mean')
    ax.set_title(col.replace('_',' ').title(), fontweight='bold')
    ax.set_ylabel('Accident Rate')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    ax.set_ylim(0, 1)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/plots/eda_accident_rates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Numeric feature distributions
num_cols = ['weekly_hours','safety_training_hrs','site_hazard_score',
            'equipment_age_yrs','prev_incidents_3yr','overtime_days_month']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Numeric Feature Distributions by Outcome', fontsize=13, fontweight='bold')

for ax, col in zip(axes.flat, num_cols):
    for val, label, color in [(0,'No Accident','#185FA5'),(1,'Accident','#E24B4A')]:
        subset = df[df['accident_occurred']==val][col]
        ax.hist(subset, bins=30, alpha=0.55, color=color,
                label=label, density=True)
    ax.set_title(col.replace('_',' ').title(), fontweight='bold')
    ax.legend(fontsize=8)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/plots/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap (numeric only)
num_df = df.select_dtypes(include='number').drop(columns=['accident_probability'])
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.4, ax=ax, cbar_kws={'shrink':0.7})
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/plots/eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 3 · Preprocessing & Feature Engineering

In [ ]:
from src.preprocessing.pipeline import prepare_data, save_preprocessor

# Save dataset first
Path('../data').mkdir(exist_ok=True)
df.to_csv('../data/occupational_accidents.csv', index=False)

data = prepare_data('../data/occupational_accidents.csv', apply_smote=True)

X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']
feat_names      = data['feature_names']
preprocessor    = data['preprocessor']

save_preprocessor(preprocessor, '../outputs/models/preprocessor.pkl')
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

## 4 · Baseline Model Training

In [ ]:
from src.models.train import get_baseline_models, train_baseline
from src.evaluation.metrics import compare_models

print('Training baseline models...')
baselines = get_baseline_models()
fitted_baselines = train_baseline(baselines, X_train, y_train)

baseline_df = compare_models(fitted_baselines, X_test, y_test)
baseline_df[['model','accuracy','f1_macro','auc_roc']]

## 5 · Hyperparameter Optimisation

In [ ]:
from src.models.train import tune_all, save_model

print('Tuning top models (this may take several minutes)...')
tuned_models = tune_all(
    X_train, y_train,
    models_to_tune=['Random Forest', 'XGBoost', 'LightGBM'],
    n_iter=40,
)

Path('../outputs/models').mkdir(parents=True, exist_ok=True)
for name, model in tuned_models.items():
    slug = name.replace(' ','_')
    save_model(model, f'../outputs/models/tuned_{slug}.pkl', name)

## 6 · Model Evaluation & Comparison

In [ ]:
from src.evaluation.metrics import (
    compare_models, plot_model_comparison,
    plot_confusion_matrix, plot_roc_pr_curves,
    evaluate_model, generate_report
)

all_models = {
    **fitted_baselines,
    **{f'{k} (tuned)': v for k, v in tuned_models.items()}
}

out_plots = Path('../outputs/plots')
out_plots.mkdir(parents=True, exist_ok=True)

comparison_df = compare_models(all_models, X_test, y_test)

plot_model_comparison(comparison_df, out_plots)

best_name  = comparison_df.iloc[0]['model']
best_model = all_models[best_name]
best_metrics = evaluate_model(best_model, X_test, y_test, best_name)

plot_confusion_matrix(y_test, best_metrics['y_pred'], best_name, out_plots)
plot_roc_pr_curves(all_models, X_test, y_test, out_plots)
generate_report(comparison_df, best_metrics, Path('../outputs/reports'))

print(f'\n★  Best model: {best_name}')

## 7 · Feature Importance & SHAP

In [ ]:
from src.evaluation.metrics import plot_feature_importance, plot_shap_summary

plot_feature_importance(best_model, feat_names, best_name, out_plots, top_n=20)
plot_shap_summary(best_model, X_test, feat_names, best_name, out_plots, max_samples=500)

## 8 · Single-Record Prediction Demo

In [ ]:
from src.utils.predict import predict_single, print_result

high_risk_worker = {
    'industry_sector':    'Mining',
    'age_group':          '18-25',
    'experience_level':   '<1yr',
    'shift_type':         'Night',
    'weekly_hours':       62,
    'ppe_compliance':     'None',
    'education_level':    'Primary',
    'season':             'Winter',
    'employment_type':    'Agency',
    'safety_training_hrs':2,
    'site_hazard_score':  8.4,
    'equipment_age_yrs':  18,
    'prev_incidents_3yr': 3,
    'near_misses_3yr':    5,
    'team_size':          8,
    'overtime_days_month':12,
}

result = predict_single(high_risk_worker, best_model, preprocessor)
print_result(result)

In [ ]:
low_risk_worker = {
    'industry_sector':    'Manufacturing',
    'age_group':          '36-45',
    'experience_level':   '7-15yr',
    'shift_type':         'Day',
    'weekly_hours':       40,
    'ppe_compliance':     'Full',
    'education_level':    'Vocational',
    'season':             'Spring',
    'employment_type':    'Permanent',
    'safety_training_hrs':24,
    'site_hazard_score':  3.2,
    'equipment_age_yrs':  4,
    'prev_incidents_3yr': 0,
    'near_misses_3yr':    0,
    'team_size':          22,
    'overtime_days_month':1,
}

result = predict_single(low_risk_worker, best_model, preprocessor)
print_result(result)